In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
from google.colab import files

uploaded = files.upload()

Saving Dataset .csv to Dataset  (1).csv


In [7]:
import pandas as pd

filename = next(
    name for name in uploaded.keys()
    if name.lower().endswith(".csv")
)

df = pd.read_csv(filename)

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset Shape: (9551, 21)


,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [8]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Dataset Shape: (9551, 21)

Column Names:
['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address', 'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines', 'Average Cost for two', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu', 'Price range', 'Aggregate rating', 'Rating color', 'Rating text', 'Votes']

Missing Values:
Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                9
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64

Duplicate Rows: 0


In [9]:
relevant_columns = [
    "Restaurant Name",
    "Cuisines",
    "Price range",
    "Aggregate rating",
    "City",
    "Has Online delivery",
    "Has Table booking"
]

available_columns = [
    col for col in relevant_columns
    if col in df.columns
]

restaurants = df[available_columns].copy()

print("Selected Columns:")
print(restaurants.columns.tolist())

restaurants.head()

Selected Columns:
['Restaurant Name', 'Cuisines', 'Price range', 'Aggregate rating', 'City', 'Has Online delivery', 'Has Table booking']


,Restaurant Name,Cuisines,Price range,Aggregate rating,City,Has Online delivery,Has Table booking
0,Le Petit Souffle,"French, Japanese, Desserts",3,4.8,Makati City,No,Yes
1,Izakaya Kikufuji,Japanese,3,4.5,Makati City,No,Yes
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",4,4.4,Mandaluyong City,No,Yes
3,Ooma,"Japanese, Sushi",4,4.9,Mandaluyong City,No,No
4,Sambo Kojin,"Japanese, Korean",4,4.8,Mandaluyong City,No,Yes


In [10]:
restaurants["Cuisines"] = (
    restaurants["Cuisines"].fillna("").astype(str)
)

restaurants["Restaurant Name"] = (
    restaurants["Restaurant Name"]
    .fillna("Unknown Restaurant")
    .astype(str)
)

restaurants["City"] = (
    restaurants["City"].fillna("Unknown").astype(str)
)

restaurants["Aggregate rating"] = pd.to_numeric(
    restaurants["Aggregate rating"], errors="coerce"
)

restaurants["Price range"] = pd.to_numeric(
    restaurants["Price range"], errors="coerce"
)

restaurants = restaurants.dropna(
    subset=["Aggregate rating", "Price range"]
).reset_index(drop=True)

print("Dataset after preprocessing:", restaurants.shape)
print("\nRemaining missing values:")
print(restaurants.isnull().sum())

Dataset after preprocessing: (9551, 7)

Remaining missing values:
Restaurant Name        0
Cuisines               0
Price range            0
Aggregate rating       0
City                   0
Has Online delivery    0
Has Table booking      0
dtype: int64


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

print("Recommendation libraries imported successfully!")


Recommendation libraries imported successfully!


In [14]:
# Convert cuisine text into numerical feature vectors
tfidf = TfidfVectorizer()

cuisine_matrix = tfidf.fit_transform(restaurants["Cuisines"])

print("Cuisine matrix shape:", cuisine_matrix.shape)
print("Cuisine features created successfully!")


Cuisine matrix shape: (9551, 150)
Cuisine features created successfully!


In [15]:
similarity_matrix = cosine_similarity(cuisine_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)
print("Restaurant similarity calculated successfully!")

Similarity matrix shape: (9551, 9551)
Restaurant similarity calculated successfully!


In [17]:

def recommend_restaurants(
    preferred_cuisine,
    max_price=3,
    min_rating=4.0,
    city=None,
    top_n=5
):
    cuisine_mask = restaurants["Cuisines"].str.contains(
        preferred_cuisine,
        case=False,
        regex=False,
        na=False
    )

    filtered = restaurants[
        cuisine_mask
        & (restaurants["Price range"] <= max_price)
        & (restaurants["Aggregate rating"] >= min_rating)
    ].copy()

    if city:
        filtered = filtered[
            filtered["City"].str.contains(
                city, case=False, regex=False, na=False
            )
        ].copy()

    if filtered.empty:
        print("No restaurants found. Try changing your preferences.")
        return pd.DataFrame()

    # Compare each candidate with the preferred cuisine
    cuisine_vector = tfidf.transform([preferred_cuisine])
    scores = cosine_similarity(
        cuisine_vector,
        cuisine_matrix[filtered.index]
    ).flatten()

    filtered["Similarity Score"] = scores

    recommendations = filtered.sort_values(
        ["Similarity Score", "Aggregate rating"],
        ascending=False
    ).head(top_n)

    return recommendations[
        [
            "Restaurant Name",
            "Cuisines",
            "City",
            "Price range",
            "Aggregate rating",
            "Has Online delivery",
            "Has Table booking",
            "Similarity Score"
        ]
    ].reset_index(drop=True)


print("Recommendation function is ready!")


Recommendation function is ready!


In [18]:
print("Italian Cuisine Recommendations:")
display(
    recommend_restaurants(
        preferred_cuisine="Italian",
        max_price=3,
        min_rating=4.0,
        top_n=5
    )
)

print("Indian Cuisine Recommendations:")
display(
    recommend_restaurants(
        preferred_cuisine="Indian",
        max_price=2,
        min_rating=3.5,
        top_n=5
    )
)


Italian Cuisine Recommendations:


,Restaurant Name,Cuisines,City,Price range,Aggregate rating,Has Online delivery,Has Table booking,Similarity Score
0,Fusilli Reasons,Italian,Chennai,1,4.6,No,No,1.0
1,Cafe Parmesan,Italian,Faridabad,3,4.5,No,No,1.0
2,La Favorita,Italian,Edinburgh,3,4.5,No,No,1.0
3,Bocca Di Lupo,Italian,London,3,4.5,No,No,1.0
4,Flying Spaghetti Monster,Italian,Vizag,3,4.4,No,No,1.0


Indian Cuisine Recommendations:


,Restaurant Name,Cuisines,City,Price range,Aggregate rating,Has Online delivery,Has Table booking,Similarity Score
0,Mother India's Cafe,Indian,Edinburgh,2,4.4,No,No,1.0
1,Rasoi Ghar,Indian,Dubai,2,4.3,Yes,Yes,1.0
2,Gokul Gujarati Restaurant,Indian,Doha,2,4.3,No,No,1.0
3,Taste of India,Indian,Cedar Rapids/Iowa City,2,4.2,No,No,1.0
4,Akbars,Indian,Manchester,2,4.2,No,No,1.0


In [20]:
# Test a stricter budget and rating preference
test_results = recommend_restaurants(
    preferred_cuisine="Japanese",
    max_price=2,
    min_rating=4.5,
    top_n=5
)

print("Number of recommendations:", len(test_results))
display(test_results)


Number of recommendations: 3


,Restaurant Name,Cuisines,City,Price range,Aggregate rating,Has Online delivery,Has Table booking,Similarity Score
0,Marukame Udon,Japanese,Rest of Hawaii,1,4.9,No,No,1.000000
1,Miyabi 9,"Japanese, Sushi",Des Moines,2,4.8,No,No,0.669067
2,Kobe Hibachi & Sushi,"American, Japanese, Sushi",Dalton,2,4.6,No,No,0.591235


In [22]:
from google.colab import files
import os

os.makedirs("outputs", exist_ok=True)

test_results.to_csv(
    "outputs/japanese_restaurant_recommendations.csv",
    index=False
)

print("Recommendation results saved successfully!")

files.download("outputs/japanese_restaurant_recommendations.csv")


Recommendation results saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
print("Conclusion:")
print(
    "A content-based restaurant recommendation system was "
    "developed using TF-IDF and cosine similarity. "
    "The system recommends restaurants according to cuisine "
    "preferences, price range, and minimum aggregate rating."
)

Conclusion:
A content-based restaurant recommendation system was developed using TF-IDF and cosine similarity. The system recommends restaurants according to cuisine preferences, price range, and minimum aggregate rating.
